### Build End-to-End Pipeline Dataset: 
### Orders data with: 1)NULL values 2)String salary 3)Date column 
### Tasks: 1)Clean NULLs 2)Cast columns 3)Add derived columns 4)Apply UDF 
### Load: 1)Full load 2)Incremental load 3)Parameterize notebook

### 1. SAMPLE DATASET (Dirty + Realistic)
### 1)NULLs 2)String salary 3)Dates 4)Updates (for incremental load)
### from pyspark.sql import SparkSession

### spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

### data = [
###     (1, "C001", "Laptop", "50000", "2024-01-01"),
###    (2, "C002", "Mobile", None, "2024-01-02"),
###    (3, "C003", "Tablet", "20000", "2024-01-03"),
###    (4, "C004", "Laptop", "55000", "2024-01-04"),
###    (5, "C005", "Headphones", None, "2024-01-05"),
###    ## Dont use this for first time (3, "C003", "Tablet", "22000", "2024-01-06"),  # updated
###    (6, "C006", "Camera", "30000", "2024-01-06"),
###    (7, "C007", "Mobile", "18000", "2024-01-07"),
###    (8, "C008", "Watch", "8000", "2024-01-07")
###]

### columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

### df = spark.createDataFrame(data, columns)
### TASK LIST
### Task 1: Handle NULLs
### Replace NULL amount with 1000
### Task 2: Cast Columns
### Convert amount → integer
### Convert updated_date → date
### Task 3: Add Derived Columns
### bonus = amount * 0.1
### category:
### 20000 → High
### else → Low
### Task 4: Apply UDF
### Create amount_bucket:
### < 10000 → Low
### 10000–30000 → Medium
### 30000 → High
### Task 5: Full Load
### Load all data to target
### Task 6: Incremental Load
### Load only new/updated records
### Handle duplicates (keep latest)
### Task 7: Parameterization
### Pass:
### input path
last_loaded_date

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()
data = [
(1, "C001", "Laptop", "50000", "2024-01-01"),
(2, "C002", "Mobile", None, "2024-01-02"),
(3, "C003", "Tablet", "20000", "2024-01-03"),
(4, "C004", "Laptop", "55000", "2024-01-04"),
(5, "C005", "Headphones", None, "2024-01-05"),
## Dont use this for first time (3, "C003", "Tablet", "22000", "2024-01-06"), # updated
(6, "C006", "Camera", "30000", "2024-01-06"),
(7, "C007", "Mobile", "18000", "2024-01-07"),
(8, "C008", "Watch", "8000", "2024-01-07")
]
columns = ["order_id", "customer_id", "product", "amount", "updated_date"]
df = spark.createDataFrame(data, columns)

In [0]:
from pyspark.sql.functions import *
df = df.withColumn(
    "amount",
    when(col("amount").isNull(), 1000).otherwise(col("amount"))
)
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
## Task 2: Cast Columns
## Convert amount → integer
## Convert updated_date → date

df = df.withColumn("amount",col("amount").cast("int"))
df = df.withColumn("updated_date",col("updated_date").cast("date"))

df.display()


order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
## Task 3: Add Derived Columns
## bonus = amount * 0.1
## category:
## 20000 → High
## else → Low
df = df.withColumn("bonus",col("amount") * 0.1)
df = df.withColumn("category",when(col("amount") == 2000,"High").otherwise("Low"))

df.display()

order_id,customer_id,product,amount,updated_date,bonus,category
1,C001,Laptop,50000,2024-01-01,5000.0,Low
2,C002,Mobile,1000,2024-01-02,100.0,Low
3,C003,Tablet,20000,2024-01-03,2000.0,Low
4,C004,Laptop,55000,2024-01-04,5500.0,Low
5,C005,Headphones,1000,2024-01-05,100.0,Low
6,C006,Camera,30000,2024-01-06,3000.0,Low
7,C007,Mobile,18000,2024-01-07,1800.0,Low
8,C008,Watch,8000,2024-01-07,800.0,Low


In [0]:
## Task 4: Apply UDF
## Create amount_bucket:
## < 10000 → Low
## 10000–30000 → Medium
## 30000 → High

def amount_bucket(amount):
    if amount < 10000:
        return "Low"
    elif amount >= 10000 and amount < 30000:
        return "Medium"
    else:
        return "High"

amount_udf = udf(amount_bucket, StringType())

df = df.withColumn("amount_bucket", amount_udf(col("amount")))

df.display()

order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,Low,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,Low,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,Low,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,Low,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


In [0]:
## Task 5: Full Load
## Load all data to target

df.write.format("delta").mode("overwrite").save('/Volumes/workspace/default/databricks_practice/orders_data')

In [0]:
## Task 6: Incremental Load
## Load only new/updated records
## Handle duplicates (keep latest)

from pyspark.sql.functions import *
from pyspark.sql.window import Window

window = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

df = df.withColumn("rn", row_number().over(window)) \
       .filter(col("rn") == 1) \
       .drop("rn")
df.write.format("delta").mode("overwrite").save('/Volumes/workspace/default/databricks_practice/orders_data')

df.display()

order_id,customer_id,product,amount,updated_date,bonus,category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,Low,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,Low,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,Low,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,Low,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low
